In [16]:
import os
import time
import json
import numpy as np
import pandas as pd
import pm4py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import optuna
import warnings
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate
from tensorflow.keras.optimizers import Adam
from ACF_code import activity_context_frequency, pmi


In [17]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'subtrace': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'subtrace': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, True)
    test_df = build_prefix_df(test_split, prefix_lengths, True)

    return train_df, test_df, train_split, test_split

In [19]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
#GLOBAL_prefix_lengths = [100, 150, 200, 300, 400, 500, 600, 700, 800]

train_df_2013, test_df_2013, full_train_2013, full_test_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

parsing log, completed traces :: 100%|██████████| 7554/7554 [00:07<00:00, 956.83it/s] 


In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

class LSTMDataProcessor:
    def __init__(self):
        self.tokenizer = Tokenizer(filters='', lower=False)
        self.max_len = 0
        self.vocab_size = 0
        self.classes = None

    def fit(self, train_df, test_df):
        """
        Fits the tokenizer on the combined corpus of activities to ensure
        consistent mapping between train and test sets.
        """
        # Gather all unique activities from subtraces and next_activities
        all_traces = list(train_df['subtrace']) + list(test_df['subtrace'])
        all_next = list(train_df['next_activity']) + list(test_df['next_activity'])
        
        # We need to flatten the lists of subtraces for the tokenizer
        corpus = [act for sublist in all_traces for act in sublist] + all_next
        
        self.tokenizer.fit_on_texts(corpus)
        
        # +1 for zero padding
        self.vocab_size = len(self.tokenizer.word_index) + 1
        self.classes = sorted(list(self.tokenizer.word_index.keys()))
        
        # Calculate max length for padding (based on your longest prefix)
        max_train = train_df['subtrace'].apply(len).max()
        max_test = test_df['subtrace'].apply(len).max()
        self.max_len = max(max_train, max_test)
        
        print(f"Vocab Size: {self.vocab_size}")
        print(f"Max Sequence Length: {self.max_len}")

    def transform(self, df):
        """
        Converts the dataframe into X (padded sequences) and y (one-hot targets)
        """
        # 1. Convert lists of strings to lists of integers
        X_seq = self.tokenizer.texts_to_sequences(df['subtrace'])
        
        # 2. Pad sequences to ensure uniform length for the LSTM
        # 'pre' padding is usually better for LSTMs as the last steps are most relevant
        X_padded = pad_sequences(X_seq, maxlen=self.max_len, padding='pre')
        
        # 3. Handle Targets
        y_seq = self.tokenizer.texts_to_sequences(df['next_activity'])
        # Flatten list of lists [[1], [4]] -> [1, 4]
        y_flat = [item[0] for item in y_seq]
        
        # One-hot encode targets
        # Note: to_categorical creates a matrix of size (N, vocab_size + 1)
        # We slice [:, 1:] if we want to ignore the 0 index, but usually keeping it simple is safer
        y_encoded = to_categorical(y_flat, num_classes=self.vocab_size)
        
        return X_padded, y_encoded, y_flat

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, Input

def build_tax_model(vocab_size, input_length, embedding_dim=50, embedding_type='learned', embedding_matrix=None):
    model = Sequential()
    
    if embedding_type == 'one_hot':
        model.add(Embedding(input_dim=vocab_size, 
                            output_dim=vocab_size, 
                            input_length=input_length,
                            embeddings_initializer='identity',
                            trainable=False)) 
                            
    # CASE B: Pre-Trained (Inject your own matrix)
    elif embedding_type == 'pretrained':
        if embedding_matrix is None:
            raise ValueError("For 'pretrained' type, must provide an 'embedding_matrix'.")
            
        model.add(Embedding(input_dim=vocab_size, 
                            output_dim=embedding_matrix.shape[1], # Width of your vector
                            input_length=input_length,
                            weights=[embedding_matrix], # Load your matrix
                            trainable=False)) # Freeze weights to keep your pre-training
                            
    # CASE C: Learned (Standard Keras)
    else: 
        model.add(Embedding(input_dim=vocab_size, 
                            output_dim=embedding_dim, 
                            input_length=input_length,
                            trainable=True))

    # --- 2. LSTM Layers (Tax et al. Structure) ---
    # Layer 1: Returns sequences to feed into the next LSTM layer
    model.add(LSTM(100, return_sequences=True, dropout=0.2))
    
    # Layer 2: Returns only the last hidden state
    model.add(LSTM(100, return_sequences=False, dropout=0.2))
    
    # --- 3. Output Layer ---
    model.add(Dense(vocab_size, activation='softmax'))
    
    model.compile(loss='categorical_crossentropy', 
                  optimizer='adam', 
                  metrics=['accuracy'])
    
    return model

In [23]:


# 2. Prepare Tensors
processor = LSTMDataProcessor()
processor.fit(train_df_2013, test_df_2013)

X_train, y_train, _ = processor.transform(train_df_2013)
X_test, y_test, y_test_indices = processor.transform(test_df_2013)

print(f"Training shape: {X_train.shape}") # Should be (Num_Samples, Max_Len)

# 3. Build Model (Experimenting with Learned Embeddings)
model = build_tax_model(
    vocab_size=processor.vocab_size, 
    input_length=processor.max_len,
    embedding_dim=64,       # Dimension of the vector space
    embedding_type='one_hot' # Change to 'one_hot' to replicate strict Tax et al.
)

model.summary()

# 4. Train
# Use EarlyStopping to prevent overfitting
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    callbacks=[callback],
    verbose=1
)

# 5. Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")

Vocab Size: 5
Max Sequence Length: 150
Training shape: (273719, 150)


d:\OZP\Resource-Centric-NAP\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
  33/1711 ━━━━━━━━━━━━━━━━━━━━ 8:17 296ms/step - accuracy: 0.5991 - loss: 1.1677

KeyboardInterrupt: 

In [ ]:
from gensim.models import Word2Vec

def create_word2vec_matrix(train_df, tokenizer, embedding_dim=64):
    sentences = train_df['subtrace'].tolist()

    w2v_model = Word2Vec(sentences, vector_size=embedding_dim, window=5, min_count=1, workers=4)

    vocab_size = len(tokenizer.word_index) + 1
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    
    for word, i in tokenizer.word_index.items():
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]
        else:
            # Words not found in Word2Vec (unlikely here) are initialized as random
            embedding_matrix[i] = np.random.normal(size=(embedding_dim,))
            
    return embedding_matrix

EMBED_DIM = 64
w2v_matrix = create_word2vec_matrix(train_df_2013, processor.tokenizer, embedding_dim=EMBED_DIM)

# 3. Build Model using Pre-trained Word2Vec
model = build_tax_model(
    vocab_size=processor.vocab_size, 
    input_length=processor.max_len,
    embedding_type='pretrained', 
    embedding_matrix=w2v_matrix  # Pass the matrix here
)

model.summary()

# 4. Train as usual
history = model.fit(
    X_train, y_train,
    epochs=10, # Word2Vec often needs slightly more epochs to converge since weights are frozen
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320 (1.25 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 320 (1.25 KB)

Epoch 1/10
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 335s 194ms/step - accuracy: 0.7436 - loss: 0.6406 - val_accuracy: 0.7947 - val_loss: 0.5199
Epoch 2/10
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 329s 192ms/step - accuracy: 0.7566 - loss: 0.6113 - val_accuracy: 0.7948 - val_loss: 0.5144
Epoch 3/10
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.7574 - loss: 0.6067

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

def train_doc2vec(train_df, vector_size=64):
    tagged_data = [
        TaggedDocument(words=row['subtrace'], tags=[str(i)]) 
        for i, row in train_df.iterrows()
    ]
    
    # Train the model
    model = Doc2Vec(vector_size=vector_size, window=5, min_count=1, workers=4, epochs=20)
    model.build_vocab(tagged_data)
    model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)
    
    return model

# Train the model
d2v_model = train_doc2vec(train_df_2013, vector_size=64)

def get_doc_vectors(df, d2v_model):
    vectors = [d2v_model.infer_vector(row['subtrace']) for _, row in df.iterrows()]
    return np.array(vectors)

X_train_d2v = get_doc_vectors(train_df_2013, d2v_model)
X_test_d2v = get_doc_vectors(test_df_2013, d2v_model)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

def build_doc2vec_model(input_dim, num_classes):
    model = Sequential([
        Input(shape=(input_dim,)), # The Doc2Vec vector size (e.g., 64)
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax') # Predict next activity
    ])
    
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Build and Train
model_d2v = build_doc2vec_model(input_dim=64, num_classes=processor.vocab_size)

model_d2v.fit(
    X_train_d2v, y_train, 
    epochs=20, 
    batch_size=128, 
    validation_split=0.2
)

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertConfig, BertForMaskedLM, BertModel
from torch.optim import AdamW
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# setting up BERT vocabulary
all_activities = sorted(list(set([act for seq in full_train_2013 for act in seq])))
vocab = {act: i + 5 for i, act in enumerate(all_activities)} 
vocab['[PAD]'] = 0
vocab['[MASK]'] = 1
vocab['[CLS]'] = 2
vocab['[SEP]'] = 3
vocab['[UNK]'] = 4
inv_vocab = {v: k for k, v in vocab.items()}

def tokenize(seq, max_len):
    seq_tokens = [vocab['[CLS]']] + [vocab.get(act, vocab['[UNK]']) for act in seq[-max_len+1:]]
    
    pad_len = max_len - len(seq_tokens)
    input_ids = seq_tokens + [vocab['[PAD]']] * pad_len
    
    # create attention mask (1 for real data, 0 for pads)
    attention_mask = [1] * len(seq_tokens) + [0] * pad_len
    
    return torch.tensor(input_ids), torch.tensor(attention_mask)


class LogDataset(Dataset):
    def __init__(self, traces, max_len=150):
        self.traces = traces
        self.max_len = max_len
        
    def __len__(self):
        return len(self.traces)
    
    def __getitem__(self, idx):
        return tokenize(self.traces[idx], self.max_len)

# configuring BERT
config = BertConfig(
    vocab_size=len(vocab),
    hidden_size=128, 
    num_hidden_layers=4, 
    num_attention_heads=4,
    intermediate_size=512
)

# pre-training
def pretrain_bert(full_traces, epochs=5, batch_size=32):
    model = BertForMaskedLM(config)
    model.train()
    optimizer = AdamW(model.parameters(), lr=1e-4)
    
    dataset = LogDataset(full_traces)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    print("Starting Pre-training...")
    for epoch in range(epochs):
        total_loss = 0
        for input_ids, attention_mask in loader:
            labels = input_ids.clone()
        
            rand = torch.rand(input_ids.shape)
            
            mask_arr = (rand < 0.15) & (input_ids > 4)
            input_ids[mask_arr] = vocab['[MASK]']
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")
        
    return model.bert 

bert_encoder = pretrain_bert(full_train_2013)

def get_embeddings(df, encoder, max_len=150):
    encoder.eval()
    embeddings = []
    
    # We can also batch this for speed, but row-by-row is okay for inference if dataset is small
    with torch.no_grad():
        for _, row in df.iterrows():
            input_ids, attention_mask = tokenize(row['subtrace'], max_len)
            
            input_ids = input_ids.unsqueeze(0)
            attention_mask = attention_mask.unsqueeze(0)
            
            outputs = encoder(input_ids, attention_mask=attention_mask)
            
            last_real_idx = attention_mask.sum() - 1 
            
            emb = outputs.last_hidden_state[0, last_real_idx, :].numpy()
            embeddings.append(emb)
            
    return np.array(embeddings)


In [ ]:
def create_bert_matrix(bert_encoder, tokenizer, vocab_mapping):
    # Get the embedding weights from the PyTorch model
    # Shape: (bert_vocab_size, hidden_size)
    bert_embeddings = bert_encoder.embeddings.word_embeddings.weight.data.numpy()
    
    # Target matrix for Keras
    keras_vocab_size = len(tokenizer.word_index) + 1
    embedding_dim = bert_embeddings.shape[1]
    embedding_matrix = np.zeros((keras_vocab_size, embedding_dim))
    
    for word, i in tokenizer.word_index.items():
        # Find what ID this activity has in your BERT vocab
        if word in vocab_mapping:
            bert_id = vocab_mapping[word]
            embedding_matrix[i] = bert_embeddings[bert_id]
        else:
            # Fallback for activities not in BERT pre-training
            embedding_matrix[i] = np.random.normal(size=(embedding_dim,))
            
    return embedding_matrix

# Create the matrix
# Note: 'vocab' is the dictionary you created for BERT earlier
bert_matrix = create_bert_matrix(bert_encoder, processor.tokenizer, vocab)

# 1. Build the model with BERT weights
model_bert_lstm = build_tax_model(
    vocab_size=processor.vocab_size,
    input_length=processor.max_len,
    embedding_type='pretrained',
    embedding_matrix=bert_matrix
)

# 2. Train using the same padded sequences (X_train) from your processor
history = model_bert_lstm.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

Num GPUs Available:  0
Is built with CUDA:  False
